# Notebook 3 of 4: Business tables with Spark (Part 5)

`gold_monthly_sales` and `gold_customer_features` use the DataFrame API and `gold_category_mix` uses Spark SQL.

**My revenue rule:** cancelled and returned orders are not sales, so they are left out of revenue, order count and average order value. Pending and shipped orders count, because the customer has bought and the money is still expected.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA = "mustafa_shop"
DB = f"{CATALOG}.{SCHEMA}"

### Join the star schema
I join `dim_customer` on `customer_sk`, so `city` is the city **at the time of the order**. Joining on `customer_id` would count the orders of customers who moved twice.

In [ ]:
orders = (spark.table(f"{DB}.fact_orders")
    .join(spark.table(f"{DB}.dim_customer").select("customer_sk", "customer_id", "city", "is_current"), "customer_sk")
    .join(spark.table(f"{DB}.dim_date").select("date_key", "date", "year"), "date_key")
    .join(spark.table(f"{DB}.dim_category"), "category_key")
    .withColumn("month", F.date_format("date", "yyyy-MM")))

is_sale = ~F.col("status").isin("cancelled", "returned")
sold = orders.filter(is_sale)

print(f"All orders: {orders.count():,}   Sold orders: {sold.count():,}")

All orders: 200,000   Sold orders: 173,857


### 5.1 gold_monthly_sales (DataFrame API)

In [ ]:
gold_monthly_sales = (sold
    .groupBy("month", "city")
    .agg(F.round(F.sum("amount"), 2).alias("revenue"),
         F.count("*").alias("order_count"))
    .withColumn("avg_order_value", F.round(F.col("revenue") / F.col("order_count"), 2))
    .orderBy("month", "city"))

gold_monthly_sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.gold_monthly_sales")
display(spark.table(f"{DB}.gold_monthly_sales").limit(10))

  month       city     revenue  order_count  avg_order_value
2025-01 Faisalabad 17903229.25          660         27126.10
2025-01  Hyderabad  7666129.55          381         20121.07
2025-01  Islamabad 21564864.35          955         22581.01
2025-01    Karachi 45892679.76         1999         22957.82
2025-01     Lahore 43996710.74         1763         24955.59
2025-01     Multan 12569983.66          546         23021.95
2025-01   Peshawar 12231596.99          533         22948.59
2025-01     Quetta  5053501.55          213         23725.36
2025-01 Rawalpindi 15638387.51          693         22566.22
2025-01    Sialkot 10279273.96          419         24532.87


### 5.2 gold_category_mix (Spark SQL)
A window function gives each month's total, so the shares inside one month add up to 1. I also keep `order_count`, because Question 3 needs it.

In [ ]:
sold.createOrReplaceTempView("sold_orders")

gold_category_mix = spark.sql("""
    WITH monthly AS (
        SELECT month, category, SUM(amount) AS revenue, COUNT(*) AS order_count
        FROM sold_orders
        GROUP BY month, category
    )
    SELECT month,
           category,
           ROUND(revenue, 2) AS revenue,
           order_count,
           ROUND(revenue / SUM(revenue) OVER (PARTITION BY month), 4) AS revenue_share
    FROM monthly
    ORDER BY month, category
""")

gold_category_mix.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.gold_category_mix")
display(spark.table(f"{DB}.gold_category_mix").limit(12))

  month    category      revenue  order_count  revenue_share
2025-01      Beauty   9659881.13         1350         0.0501
2025-01       Books   4233955.13         1335         0.0220
2025-01 Electronics 107064496.92         1343         0.5553
2025-01     Fashion  18941440.38         1382         0.0982
2025-01     Grocery   6967827.80         1396         0.0361
2025-01        Home  45928755.96         1356         0.2382
2025-02      Beauty   8458816.00         1225         0.0498
2025-02       Books   3932550.42         1250         0.0232
2025-02 Electronics  91724310.14         1169         0.5405
2025-02     Fashion  14422262.60         1106         0.0850
2025-02     Grocery   6643220.92         1239         0.0391
2025-02        Home  44531621.00         1277         0.2624


### 5.3 gold_customer_features (DataFrame API)
One row per customer, so I group by `customer_id`, because a customer who moved has two `customer_sk` values.

- `total_spend` uses sold orders only. `num_orders` counts every order placed.
- `days_since_last_order` is measured from 2026-10-24, the last day in the data. Using today's date would make every customer look inactive.
- `favourite_category` is the category with the most orders. If there is a tie I take the first by name, so the result is the same every run.

In [ ]:
LAST_DATA_DAY = "2026-10-24"

per_customer = (orders
    .groupBy("customer_id")
    .agg(F.round(F.sum(F.when(is_sale, F.col("amount")).otherwise(0)), 2).alias("total_spend"),
         F.count("*").alias("num_orders"),
         F.max("date").alias("last_order_date"),
         F.round(F.avg(F.when(F.col("source") == "app", 1).otherwise(0)), 4).alias("app_order_share"))
    .withColumn("days_since_last_order", F.datediff(F.to_date(F.lit(LAST_DATA_DAY)), F.col("last_order_date"))))

category_rank = Window.partitionBy("customer_id").orderBy(F.desc("category_orders"), "category")
favourite = (orders
    .groupBy("customer_id", "category")
    .agg(F.count("*").alias("category_orders"))
    .withColumn("rank", F.row_number().over(category_rank))
    .filter("rank = 1")
    .select("customer_id", F.col("category").alias("favourite_category")))

# start from all customers, so anyone who never ordered still gets a row
all_customers = spark.table(f"{DB}.dim_customer").filter("is_current").select("customer_id", "name", "city", "segment")

gold_customer_features = (all_customers
    .join(per_customer, "customer_id", "left")
    .join(favourite, "customer_id", "left")
    .fillna({"total_spend": 0, "num_orders": 0, "app_order_share": 0})
    .select("customer_id", "name", "city", "segment", "total_spend", "num_orders", "last_order_date",
            "days_since_last_order", "favourite_category", "app_order_share"))

gold_customer_features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.gold_customer_features")

check = spark.table(f"{DB}.gold_customer_features")
print(f"Rows: {check.count():,}   Distinct customers: {check.select('customer_id').distinct().count():,}")
display(check.limit(5))

Rows: 2,000   Distinct customers: 2,000

 customer_id          name       city  segment  total_spend  num_orders last_order_date  days_since_last_order favourite_category  app_order_share
           1  Usman Sheikh    Karachi Consumer    502690.99          46      2026-10-24                      0            Grocery           0.0000
           2     Iqra Khan Faisalabad Consumer   1452276.42          56      2026-10-16                      8        Electronics           0.0179
           3   Maryam Raza Faisalabad Consumer   1979487.34          67      2026-10-24                      0        Electronics           0.0448
           4    Ali Sheikh     Lahore Consumer    514202.24          43      2026-09-20                     34            Grocery           0.0698
           5 Hamza Qureshi    Karachi Consumer   4106984.57         168      2026-10-24                      0              Books           0.0179


### Extra gold table for the rain question
The three gold tables above have no daily weather, so they cannot answer Question 2. This table has one row per city per day, for the three cities with weather. It starts from the weather table so that a day with no orders would still count. Here I count every order placed, including ones cancelled later, because the question is about how many people order.

In [ ]:
daily_orders = (orders
    .groupBy("city", "date")
    .agg(F.count("*").alias("orders_placed")))

gold_city_weather_daily = (spark.table(f"{DB}.weather_daily")
    .join(daily_orders, ["city", "date"], "left")
    .fillna({"orders_placed": 0})
    .withColumn("is_rainy", F.col("rain_mm") >= 1)     # at least 1 mm counts as a rainy day
    .select("city", "date", "max_temp", "rain_mm", "is_rainy", "orders_placed"))

gold_city_weather_daily.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.gold_city_weather_daily")
print(f"Rows: {gold_city_weather_daily.count():,}")

Rows: 1,986


# 5.4 Answers from the gold tables

### Question 1: Which city grew fastest in 2026?
The data stops on 24 October 2026, so 2026 is not a full year and October is not a full month. To be fair I compare the same nine full months in both years: January to September 2026 against January to September 2025.

In [ ]:
q1 = spark.sql(f"""
    SELECT city,
           ROUND(SUM(CASE WHEN month BETWEEN '2025-01' AND '2025-09' THEN revenue END), 0) AS revenue_2025,
           ROUND(SUM(CASE WHEN month BETWEEN '2026-01' AND '2026-09' THEN revenue END), 0) AS revenue_2026,
           SUM(CASE WHEN month BETWEEN '2025-01' AND '2025-09' THEN order_count END) AS orders_2025,
           SUM(CASE WHEN month BETWEEN '2026-01' AND '2026-09' THEN order_count END) AS orders_2026
    FROM {DB}.gold_monthly_sales
    GROUP BY city
""").withColumn("revenue_growth_pct", F.round((F.col("revenue_2026") / F.col("revenue_2025") - 1) * 100, 2)) \
    .withColumn("order_growth_pct", F.round((F.col("orders_2026") / F.col("orders_2025") - 1) * 100, 2)) \
    .orderBy(F.desc("revenue_growth_pct"))
display(q1)

      city  revenue_2025  revenue_2026  orders_2025  orders_2026  revenue_growth_pct  order_growth_pct
    Lahore   352070369.0   377232263.0        14956        15852                7.15              5.99
   Karachi   420581103.0   445602895.0        17839        18476                5.95              3.57
    Multan   113416572.0   117856791.0         4855         5047                3.91              3.95
Faisalabad   142656957.0   143120167.0         5905         5908                0.32              0.05
 Hyderabad    78206371.0    77948583.0         3398         3270               -0.33             -3.77
Rawalpindi   147152632.0   144327594.0         6049         6041               -1.92             -0.13
  Peshawar   113134695.0   109382443.0         4701         4696               -3.32             -0.11
 Islamabad   198614157.0   183985358.0         8155         7810               -7.37             -4.23
   Sialkot    84755058.0    77431363.0         3548         3375         

### Question 2: Do Lahore, Karachi and Islamabad get fewer orders on rainy days?
The rainy and dry groups have very different numbers of days, so I compare the **average orders per day**, not the totals.

In [ ]:
q2 = spark.sql(f"""
    SELECT city,
           is_rainy,
           COUNT(*) AS days,
           ROUND(AVG(orders_placed), 2) AS avg_orders_per_day,
           ROUND(STDDEV(orders_placed), 2) AS std_dev
    FROM {DB}.gold_city_weather_daily
    GROUP BY city, is_rainy
    ORDER BY city, is_rainy
""")
display(q2)

     city  is_rainy  days  avg_orders_per_day  std_dev
Islamabad     False   444               33.57     5.78
Islamabad      True   218               33.87     5.95
  Karachi     False   582               76.01     9.12
  Karachi      True    80               77.29     9.09
   Lahore     False   502               64.63     8.38
   Lahore      True   160               64.89     8.71


### Question 3: Which category has the highest average order value?
Total revenue divided by total orders across all months. Averaging the monthly averages would give every month the same weight, however many orders it had.

In [ ]:
q3 = spark.sql(f"""
    SELECT category,
           ROUND(SUM(revenue), 0) AS revenue,
           SUM(order_count) AS order_count,
           ROUND(SUM(revenue) / SUM(order_count), 2) AS avg_order_value
    FROM {DB}.gold_category_mix
    GROUP BY category
    ORDER BY avg_order_value DESC
""")
display(q3)

   category      revenue  order_count  avg_order_value
Electronics 2283952901.0        29097         78494.45
       Home 1018498199.0        29013         35104.89
    Fashion  385052323.0        28657         13436.59
     Beauty  205372159.0        29060          7067.18
    Grocery  156648012.0        29364          5334.70
      Books   92498854.0        28666          3226.78


### My answers
1. **Lahore grew fastest:** its January to September revenue was 7.15% higher in 2026 than in 2025 (orders up 5.99%), just ahead of Karachi at 5.95%.
2. **No:** average orders per day are about the same on rainy and dry days (Lahore 64.89 and 64.63, Karachi 77.29 and 76.01, Islamabad 33.87 and 33.57), and the small gaps are much less than the normal day-to-day swing of 6 to 9 orders.
3. **Electronics** has the highest average order value at 78,494, more than double Home, which is next at 35,105.